In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


# Parse date and create month column
df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df['Month'] = df['DateTime'].dt.to_period('M')

# Categorize transactions
income_types = df[df['Transaction_Amount']  > 0]['Transaction_Type'].unique().tolist()
expense_types = df[df['Transaction_Amount']  < 0]['Transaction_Type'].unique().tolist()

income_data = df[df['Transaction_Type'].isin(income_types)]
expense_data = df[df['Transaction_Type'].isin(expense_types)]

# Calculate monthly totals
monthly_income = income_data.groupby('Month')['Transaction_Amount'].sum()
monthly_expenses = expense_data.groupby('Month')['Transaction_Amount'].sum().abs()

# ===== REUSABLE FUNCTIONS =====

def autopct_format(values):
    """Create custom autopct function to show both percentage and amount"""
    def inner_autopct(pct):
        total = sum(values)
        val = int(round(pct * total / 100.0))
        return f"{pct:.1f}%\n{val:,.0f} KRW"
    return inner_autopct

def style_pie_text_white(autotexts, fontsize=10):
    """Apply white color styling to pie chart text labels"""
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
        autotext.set_fontsize(fontsize)


# Create subplots for each month
months = monthly_income.index
n_months = len(months)
cols = 3
rows = (n_months + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, 6*rows))
fig.suptitle('Monthly Cash Flow Distribution - Income vs Expenses', fontsize=16, fontweight='bold')

if rows == 1:
    axes = [axes] if n_months == 1 else axes
else:
    axes = axes.flatten()

colors = ['#2E8B57', '#DC143C']  # Green for income, Red for expenses
labels = ['Income', 'Expenses']

for i, month in enumerate(months):
    income = monthly_income.get(month, 0)
    expenses = monthly_expenses.get(month, 0)
    
    if income > 0 or expenses > 0:
        sizes = [income, expenses]
        
        # Create pie chart with amounts
        wedges, texts, autotexts = axes[i].pie(
            sizes, 
            labels=labels, 
            colors=colors,
            autopct=autopct_format(sizes),
            startangle=90,
            explode=(0.05, 0.05)
        )
        
        # Apply white text styling using reusable function
        style_pie_text_white(autotexts, fontsize=10)
            
        axes[i].set_title(f'{month}', fontweight='bold', fontsize=12)
    else:
        axes[i].text(0.5, 0.5, 'No Data', ha='center', va='center', 
                    transform=axes[i].transAxes, fontsize=12)
        axes[i].set_title(f'{month}', fontweight='bold', fontsize=12)

# Hide unused subplots
for i in range(n_months, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.savefig('visuals/monthly_cash_flow_breakdown.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Monthly Cash Flow Breakdown saved as 'monthly_cash_flow_breakdown.png'")

# Print summary
print("\n=== MONTHLY BREAKDOWN SUMMARY ===")
for month in months:
    income = monthly_income.get(month, 0)
    expenses = monthly_expenses.get(month, 0)
    net = income - expenses
    print(f"{month}: Income {income:,.0f} KRW | Expenses {expenses:,.0f} KRW | Net {net:,.0f} KRW")

In [ ]:
# ===== PLOT 1: Transaction Count by Type (VERTICAL Bar Chart) =====
# Sort by count (descending)
count_sorted = transaction_stats.sort_values('Count', ascending=False)

# Get colors for sorted transaction types
colors_plot1 = [type_colors.get(t, '#808080') for t in count_sorted.index]

plt.figure(figsize=(12, 7))
bars1 = plt.bar(range(len(count_sorted)), count_sorted['Count'], 
                color=colors_plot1, alpha=0.85, edgecolor='white', linewidth=1.5)
plt.title('Transaction Count by Type', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Number of Transactions', fontsize=13, fontweight='bold')
plt.xlabel('Transaction Type', fontsize=13, fontweight='bold')
plt.xticks(range(len(count_sorted)), count_sorted.index, rotation=45, ha='right', fontsize=11)
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.gca().set_axisbelow(True)

# Add value labels on bars
for i, v in enumerate(count_sorted['Count']):
    plt.text(i, v + max(count_sorted['Count'])*0.02, f'{int(v):,}', 
             ha='center', va='bottom', fontweight='bold', fontsize=11)

# Format y-axis to remove scientific notation
ax1 = plt.gca()
ax1.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.savefig('visuals/transaction_count_by_type.png', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.show()
print("✓ Saved: transaction_count_by_type.png")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

# Parse date and create month column
df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df['Month'] = df['DateTime'].dt.to_period('M')

# 1) Cash Flow Core KPIs
# Categorize transactions
income_types = df[df['Transaction_Amount']  > 0]['Transaction_Type'].unique().tolist()
expense_types = df[df['Transaction_Amount']  < 0]['Transaction_Type'].unique().tolist()

income_data = df[df['Transaction_Type'].isin(income_types)]
expense_data = df[df['Transaction_Type'].isin(expense_types)]

# Calculate totals
total_income = income_data['Transaction_Amount'].sum()
total_expenses = abs(expense_data['Transaction_Amount'].sum())  # Make positive
net_flow = total_income - total_expenses

print("=== CASH FLOW CORE KPIs ===")
print(f"Total Income: {total_income:,.0f} KRW")
print(f"Total Expenses: {total_expenses:,.0f} KRW") 
print(f"Net Flow: {net_flow:,.0f} KRW")

# 2) Monthly Trends
monthly_income = income_data.groupby('Month')['Transaction_Amount'].sum()
monthly_expenses = expense_data.groupby('Month')['Transaction_Amount'].sum().abs()
monthly_net = monthly_income - monthly_expenses

print("\n=== MONTHLY TRENDS ===")
for month in monthly_income.index:
    income = monthly_income.get(month, 0)
    expenses = monthly_expenses.get(month, 0)
    net = income - expenses
    print(f"{month}: Income {income:,.0f} KRW | Expenses {expenses:,.0f} KRW | Net {net:,.0f} KRW")

# Create visuals directory
os.makedirs('visuals', exist_ok=True)

# ===== PLOT 1: Overall Cash Flow Pie Chart =====
fig1, ax1 = plt.subplots(figsize=(10, 8))

sizes = [total_income, total_expenses]
labels = ['Income', 'Expenses']
colors = ['#0b5394', '#990000']
explode = (0.05, 0.05)

ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', 
        startangle=90, explode=explode, shadow=True)
ax1.set_title('Overall Cash Flow Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('visuals/cash_flow_pie_chart.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Pie chart saved as 'cash_flow_pie_chart.png'")

# ===== PLOT 2: Monthly Horizontal Bar Chart =====
fig2, ax2 = plt.subplots(figsize=(12, 8))

months = [str(m) for m in monthly_income.index]
y = np.arange(len(months))
height = 0.35

bars1 = ax2.barh(y + height/2, monthly_income.values, height, 
                 label='Income', color='#0b5394', alpha=0.8)
bars2 = ax2.barh(y - height/2, monthly_expenses.values, height, 
                 label='Expenses', color='#990000', alpha=0.8)

# Add labels for income bars (inside if long, outside if short)
max_income = max(monthly_income.values)
for bar in bars1:
    width = bar.get_width()
    # If bar is long enough (> 15% of max), put label inside
    if width > max_income * 0.15:
        ax2.text(width * 0.95, bar.get_y() + bar.get_height()/2, 
                 f'{width:,.0f}',
                 ha='right', va='center', color='white', fontweight='bold', fontsize=10)
    else:
        # Put label outside for short bars
        ax2.text(width + max_income * 0.02, bar.get_y() + bar.get_height()/2, 
                 f'{width:,.0f}',
                 ha='left', va='center', color='#0b5394', fontweight='bold', fontsize=10)

# Add labels for expense bars (inside if long, outside if short)
max_expense = max(monthly_expenses.values)
for bar in bars2:
    width = bar.get_width()
    # If bar is long enough (> 15% of max), put label inside
    if width > max_expense * 0.15:
        ax2.text(width * 0.95, bar.get_y() + bar.get_height()/2, 
                 f'{width:,.0f}',
                 ha='right', va='center', color='white', fontweight='bold', fontsize=10)
    else:
        # Put label outside for short bars
        ax2.text(width + max_expense * 0.02, bar.get_y() + bar.get_height()/2, 
                 f'{width:,.0f}',
                 ha='left', va='center', color='#990000', fontweight='bold', fontsize=10)

ax2.set_title('Monthly Income vs Expenses', fontsize=14, fontweight='bold')
ax2.set_xlabel('Amount (KRW)', fontsize=12)
ax2.set_ylabel('Month', fontsize=12)
ax2.set_yticks(y)
ax2.set_yticklabels(months)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('visuals/monthly_income_expenses_bar.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Horizontal bar chart saved as 'monthly_income_expenses_bar.png'")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

# Create visuals directory
os.makedirs('visuals', exist_ok=True)

# Font setup
import platform
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = ['AppleGothic', 'Arial Unicode MS']
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = ['Malgun Gothic']
plt.rcParams['axes.unicode_minus'] = False

# Load Data
df = pd.read_csv('transactions.csv')
df['Transaction_Amount'] = pd.to_numeric(df['Transaction_Amount'])
df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df['Month'] = df['DateTime'].dt.to_period('M')

# Data Prep
income_types = df[df['Transaction_Amount'] > 0]['Transaction_Type'].unique().tolist()
expense_types = df[df['Transaction_Amount'] < 0]['Transaction_Type'].unique().tolist()

income_data = df[df['Transaction_Type'].isin(income_types)]
expense_data = df[df['Transaction_Type'].isin(expense_types)]

total_income = income_data['Transaction_Amount'].sum()
total_expenses = abs(expense_data['Transaction_Amount'].sum())

# Monthly Data
monthly_income = income_data.groupby('Month')['Transaction_Amount'].sum()
monthly_expenses = expense_data.groupby('Month')['Transaction_Amount'].sum().abs()
all_months = sorted(list(set(monthly_income.index) | set(monthly_expenses.index)))
monthly_income = monthly_income.reindex(all_months).fillna(0)
monthly_expenses = monthly_expenses.reindex(all_months).fillna(0)

# --- CHART 1: Overall Cash Flow (Unchanged) ---
plt.figure(figsize=(10, 6))
sizes = [total_income, total_expenses]
labels = ['Income', 'Expenses']
colors = ['#2E8B57', '#DC143C']
explode = (0.05, 0.05)

wedges, texts, autotexts = plt.pie(sizes, labels=labels, colors=colors, 
                                   autopct=lambda pct: f'{pct:.1f}%\n{int(pct/100.*sum(sizes)):,} KRW',
                                   startangle=90, explode=explode, shadow=True)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(11)

plt.title('Overall Cash Flow Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visuals/overall_cash_flow.png', dpi=300, bbox_inches='tight')
plt.show()

# --- CHART 2: Monthly Trends (Horizontal Bar) ---
plt.figure(figsize=(10, 7))  # Slightly taller to accommodate months vertically

months = [str(m) for m in all_months]
y = np.arange(len(months))
height = 0.35  # Thickness of the bars

# Create Horizontal Bars (barh)
rects1 = plt.barh(y - height/2, monthly_income.values, height, label='Income', color='#2E8B57', alpha=0.9)
rects2 = plt.barh(y + height/2, monthly_expenses.values, height, label='Expenses', color='#DC143C', alpha=0.9)

plt.title('Monthly Income vs Expenses', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Amount (KRW)')
plt.ylabel('Month')
plt.yticks(y, months)
plt.legend()
plt.grid(True, alpha=0.3, axis='x')

# Format X-axis with commas
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: format(int(x), ',')))

# Function to add white labels inside the bars
def add_labels(rects):
    for rect in rects:
        width = rect.get_width()
        if width > 0:  # Only label if there is a value
            # Position text slightly inside the end of the bar
            plt.text(width * 0.95, rect.get_y() + rect.get_height()/2, 
                     f'{int(width):,}', 
                     ha='right', va='center', 
                     color='white', fontweight='bold', fontsize=10)

add_labels(rects1)
add_labels(rects2)

plt.tight_layout()
plt.savefig('visuals/monthly_trends_horizontal.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: monthly_trends_horizontal.png")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Read the data
df = pd.read_csv('transactions.csv')

print(df.columns)

# Combine Date and Time columns, then parse
df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df['Hour'] = df['DateTime'].dt.hour

# Count transactions per hour
hourly_counts = df['Hour'].value_counts().sort_index()

# Create histogram
plt.figure(figsize=(14, 8))
plt.bar(hourly_counts.index, hourly_counts.values, color='#2E8B57', alpha=0.7)

plt.title('Transaction Frequency by Hour of Day', fontsize=16, fontweight='bold')
plt.xlabel('Hour of Day (24-hour format)', fontsize=14)
plt.ylabel('Number of Transactions', fontsize=14)

# Set x-axis to show all 24 hours
plt.xticks(range(0, 24))
plt.xlim(-0.5, 23.5)
plt.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, v in hourly_counts.items():
    plt.text(i, v + 0.5, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('visuals/hourly_transaction_histogram.png', dpi=300, bbox_inches='tight')
plt.show()